In [ ]:
from scipy.stats import mannwhitneyu, kendalltau

def ordinal_informative_missingness(data, status_col, target_col, verbose=True):
    # --- FEATURE PREPARATION ---
    # Strip '%' and convert to 0.0-1.0 scale
    numeric_col = pd.to_numeric(
        data[status_col].astype(str).str.replace("%", "", regex=False), errors='coerce'
    ) / 100
    data[f"{status_col}_numeric"] = numeric_col

    # --- GROUP SEGMENTATION ---
    known = data.loc[data[status_col].notna(), target_col].dropna()
    unknown = data.loc[data[status_col].isna(), target_col].dropna()
    n1, n2 = len(known), len(unknown)

    if n1 < 2 or n2 < 2:
        if verbose: print(f"\n[SKIPPED] {status_col}: Not enough data.\n")
        return data

    # --- STATISTICAL CALCULATIONS ---
    mwu_stat, mwu_p = mannwhitneyu(known, unknown, alternative="two-sided")
    r_rb = 1 - (2 * mwu_stat) / (n1 * n2) # Effect size for MWU

    valid_idx = data[f"{status_col}_numeric"].notna() & data[target_col].notna()
    if valid_idx.sum() > 2:
        tau, tau_p = kendalltau(data.loc[valid_idx, f"{status_col}_numeric"], data.loc[valid_idx, target_col])
    else:
        tau, tau_p = np.nan, np.nan

    # --- FORMATTING UTILITIES ---
    def interpret_effect(r):
        if pd.isna(r): return "N/A"
        r_abs = abs(r)
        if r_abs < 0.1: return "Negligible"
        elif r_abs < 0.3: return "Small"
        elif r_abs < 0.5: return "Moderate"
        else: return "Large"

    def get_sig_label(p):
        if pd.isna(p): return "N/A"
        if p < 0.001: return "Significant (p < .001) ***"
        if p < 0.01:  return "Significant (p < .01)  **"
        if p < 0.05:  return "Significant (p < .05)  *"
        return "Not Significant"

    # --- VERBOSE REPORTING ---
    if verbose:
        print("\n" + "="*75)
        print(f" STATISTICAL REPORT: {status_col}")
        print("="*75)
        print(f"Group Sizes | Known: {n1:,} | Missing: {n2:,}")
        print("-" * 75)
        print(f"{'Metric':<25} | {'Value':<12} | {'Interpretation'}")
        print("-" * 75)

        # MWU Section (Missingness)
        print(f"{'MWU p-value':<25} | {mwu_p:<12.2e} | {get_sig_label(mwu_p)}")
        print(f"{'Rank-biserial (r)':<25} | {r_rb:<12.4f} | {interpret_effect(r_rb)} Effect")
        print("-" * 75)

        # Kendall Section (Ordinal Trend)
        print(f"{'Kendall’s Tau':<25} | {tau:<12.4f} | {interpret_effect(tau)} Association")
        print(f"{'Tau p-value':<25} | {tau_p:<12.2e} | {get_sig_label(tau_p)}")
        print("="*75)
        
        print("Conclusion:")
        # Corrected Logic: Missingness Check
        missing_impact = "is INFORMATIVE" if mwu_p < 0.05 else "is likely RANDOM"
        print(f" - Data missingness {missing_impact} (r = {r_rb:.4f})")
        
        # Corrected Logic: Ordinal Trend Check
        trend_impact = interpret_effect(tau).lower()
        sig_impact = "SIGNIFICANT" if tau_p < 0.05 else "not significant"
        print(f" - The {trend_impact} ordinal relationship is {sig_impact} for {target_col}")
        print("")

    return data